In [1]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Import libraries
import os
import torch
import torchaudio
from tqdm import tqdm
from denoiser import pretrained
from denoiser.dsp import convert_audio

In [3]:
# Configure paths and parameters
INPUT_ROOT = r"E:/Work/My Papers/Dravidian Lang Tech 2026/Dialect based speech processing/Dataset/Test_processed_1"
OUTPUT_ROOT = r"E:/Work/My Papers/Dravidian Lang Tech 2026/Dialect based speech processing/Dataset/Test_processed_2"
TARGET_SR = 16000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
# Load pretrained Facebook Denoiser model (dns64 recommended for best quality)
model = pretrained.dns64().to(DEVICE)
model.eval()
print("✓ Facebook Denoiser DNS64 loaded successfully")

✓ Facebook Denoiser DNS64 loaded successfully


In [5]:
# Collect all wav files
wav_files = []
for root, _, files in os.walk(INPUT_ROOT):
    for file in files:
        if file.lower().endswith(".wav"):
            wav_files.append(os.path.join(root, file))
print(f"Found {len(wav_files)} wav files")

Found 579 wav files


In [6]:
# Process all files with Facebook Denoiser
with torch.no_grad():
    for input_path in tqdm(wav_files, desc="Denoising", unit="file"):
        rel_path = os.path.relpath(os.path.dirname(input_path), INPUT_ROOT)
        output_dir = os.path.join(OUTPUT_ROOT, rel_path)
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, os.path.basename(input_path))
        
        wav, sr = torchaudio.load(input_path)
        wav = convert_audio(wav, sr, model.sample_rate, model.chin)
        wav = wav.to(DEVICE)
        
        denoised = model(wav[None])[0]
        denoised = denoised.cpu()
        
        torchaudio.save(output_path, denoised, model.sample_rate)

Denoising: 100%|██████████| 579/579 [00:38<00:00, 14.94file/s]


In [7]:
# Verify output
output_files = [f for r, _, fs in os.walk(OUTPUT_ROOT) for f in fs if f.endswith('.wav')]
print(f"✓ Processed {len(output_files)} files successfully")

✓ Processed 579 files successfully
